# Chicago Crimes: Bronze-Ingestion

Dieses Notebook lädt einen Lauf über die Socrata-API der Stadt Chicago. Es
verwendet die produktive Ingestion aus `chicago_pipeline.download_chicago`,
damit Notebook und CLI nicht zwei unterschiedliche Implementierungen besitzen —
jede Zelle ruft genau die Funktion auf, die auch
`python -m chicago_pipeline.download_chicago` benutzt.

**Pipeline-Stufe:** Socrata API → unveränderte Raw-Antworten (gzip) + abgeleitete
Bronze-Schicht (Parquet), beide mit eigenem Manifest.

Ein Lauf ist **unveränderlich**: Er wird einmal geschrieben und nie wieder
angefasst. Spätere Läufe legen neue Verzeichnisse an, statt bestehende zu
korrigieren.

## Schritt 1 – Projektpfad und Imports

Die Zelle sucht den Projekt-Root sowohl beim Start im Root als auch direkt aus
dem Notebook-Ordner. Ist das Projekt editierbar installiert
(`pip install -r requirements.txt`), wäre der `sys.path`-Eintrag nicht nötig —
er ist die Absicherung für den Fall, dass das Notebook ohne Installation
geöffnet wird.

`logging` wird eingeschaltet, weil die Ingestion ihren Fortschritt über das
`logging`-Modul nach stderr meldet: Seitenzahlen, festgeschriebene ids und
Warnungen erscheinen dadurch unter den Zellen.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json
import logging
import shutil
import sys
import tempfile

# Das Paket liegt unter src/crime/. Gesucht wird der Projekt-Root sowohl beim
# Start im Root als auch direkt aus dem Notebook-Ordner.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src' / 'crime' / 'chicago_pipeline').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src' / 'crime' / 'chicago_pipeline').exists():
    raise RuntimeError('Projekt-Root mit src/crime/chicago_pipeline nicht gefunden')

sys.path.insert(0, str(PROJECT_ROOT / 'src' / 'crime'))

from chicago_pipeline import download_chicago as dc

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(message)s',
    stream=sys.stderr,
    force=True,
)

print('Modul        :', dc.__file__)
print('Datensatz    :', dc.DATASET_ID)
print('API          :', dc.API_URL)
print('Schichten    :', dc.LAYERS)
print('Seitengrösse :', f'{dc.PAGE_SIZE:,} Zeilen ({dc.ID_PAGE_SIZE:,} bei reinen id-Abfragen)')
print('Lookback     :', dc.LOOKBACK_DAYS, 'Tage')

## Schritt 2 – Lauf konfigurieren

`MAX_RECORDS` entscheidet über die Art des Laufs:

| | `MAX_RECORDS = 5` | `MAX_RECORDS = None` |
| --- | --- | --- |
| Zweck | Testlauf, wenige Sekunden | echter Lauf über den ganzen Jahrgang |
| Vollständigkeitsprüfungen | werden **übersprungen** | laufen vollständig |
| Wasserzeichen | wird **ignoriert** | wird gesetzt |
| Zielordner | temporäres Verzeichnis | `data/` im Projekt |

Ein Testlauf kürzt absichtlich und könnte deshalb nie beweisen, dass er
vollständig ist. Damit er das Wasserzeichen nicht verfälscht, wird er im
Manifest über `max_records` markiert und später übersprungen.

Testläufe schreiben bewusst **ausserhalb** des Projekts, damit `data/` sauber
bleibt: `data/` ist nur teilweise von `.gitignore` erfasst, ein Testlauf im
falschen Ordner landet sonst schnell im Repository. Aufgeräumt wird dieser
Ordner in Schritt 10.

`FORCE_FULL` entspricht dem CLI-Schalter `--full`: Es erzwingt einen Vollabzug,
obwohl ein Wasserzeichen vorliegt. Das alte Wasserzeichen bleibt dabei erhalten.

In [ ]:
YEAR = dc.DEFAULT_YEAR        # im PoC festgelegtes Jahr
MAX_RECORDS = 5               # None = echter Lauf über den ganzen Jahrgang
USE_TOKEN = True              # False entspricht --no-token
FORCE_FULL = False            # True entspricht --full
RUN_DOWNLOAD = True           # auf False setzen, um nur zu planen

if MAX_RECORDS is None:
    DATA_ROOT = PROJECT_ROOT / 'data'
else:
    DATA_ROOT = Path(tempfile.gettempdir()) / 'chicago-ingestion-test'

print('Jahrgang   :', YEAR)
print('Zielordner :', DATA_ROOT)
print('Testlauf   :', MAX_RECORDS is not None, f'(max_records={MAX_RECORDS})')

## Schritt 3 – Wasserzeichen lesen und Modus bestimmen

Hier steckt die Delta-Logik. Die Ingestion führt **keinen** eigenen Zustand:
kein State-File, keine Datenbank. Das Wasserzeichen wird bei jedem Start aus den
Manifesten der bereits geschriebenen Läufe rekonstruiert.

**Wie `previous_watermark()` liest.** Es durchsucht alle Bronze-Manifeste unter
`bronze/city=chicago/ingest_date=*/run_*/manifest.json` und nimmt das höchste
`watermark.maximum_seen`. Ein Lauf zählt nur, wenn er drei Bedingungen erfüllt:

| Bedingung | Zweck |
| --- | --- |
| `status == "complete"` | abgebrochene Läufe zählen nicht |
| `event_year == YEAR` | jeder Jahrgang hat sein eigenes Wasserzeichen |
| `max_records is None` | Testläufe verfälschen es nicht |

**Warum nur Bronze?** Bronze wird zuletzt veröffentlicht. Das Wasserzeichen
rückt also erst vor, wenn der ganze Lauf steht. Scheitert ein Lauf zwischen den
Schichten, bleibt Raw als Beweisstück liegen, das Fenster wird aber wiederholt —
eine Wiederholung ist billiger als eine Lücke.

**Der Lookback.** `delta_start()` zieht vom Wasserzeichen **14 Tage ab**
(`LOOKBACK_DAYS`) und entfernt die Zeitzone, weil SoQL mit zonenlosen
"floating timestamps" arbeitet. Die Überlappung fängt Datensätze ab, die
rückwirkend korrigiert wurden. Doppelte Zeilen sind dabei ausdrücklich in Kauf
genommen: Mehrere Versionen derselben `id` werden erst oberhalb der
Bronze-Schicht aufgelöst.

**Zwei Zeitachsen.** Die `WHERE`-Klausel filtert gleichzeitig über das
**Tatdatum** (`date`, der Jahrgang) und im Delta zusätzlich über die
**Änderungszeit** (`updated_on`, das Delta-Fenster). Ein Delta-Lauf holt also nur
Datensätze desselben Jahrgangs, die seit dem Fensterstart geändert wurden.

In [ ]:
watermark = dc.previous_watermark(DATA_ROOT, YEAR)
updated_from = dc.delta_start(watermark) if watermark and not FORCE_FULL else None
mode = 'delta' if updated_from else 'full_snapshot'

print('bisheriges Wasserzeichen :', watermark)
print('Fensterstart (query_from):', updated_from)
print('Modus                    :', mode)
print('Lookback                 :', dc.LOOKBACK_DAYS if updated_from else 0, 'Tage')
print()
print('WHERE:')
print(' ', dc.where_clause(YEAR, updated_from))
print()
print('Erste Seite (ohne Token, so steht sie auch im Manifest):')
print(' ', dc.query_url(YEAR, updated_from, dc.PAGE_SIZE))

## Schritt 4 – Quelle prüfen: Fingerabdruck und erwartete Zeilenzahl

Vor dem Download werden zwei Dinge von der Quelle geholt:

**Schema-Fingerabdruck.** `source_schema()` liest die Feldtypen aus den
Metadaten und bildet daraus einen SHA-256. Ändert die Stadt den Typ eines
Feldes, ändert sich der Hash — auch dann, wenn die Ingestion weiterläuft. Die
Spalte `_extra` fängt zwar neue Felder auf, eine Typänderung an einem
bestehenden Feld würde sie aber nicht bemerken. Fällt der Metadaten-Endpunkt
aus, läuft der Lauf ohne Fingerabdruck weiter: Das ist Beobachtung, kein
Vollständigkeits-Gate.

**Erwartete Zeilenzahl.** `expected_rows()` fragt `count(1)` mit genau derselben
`WHERE`-Klausel ab, die auch der Download nutzt — im Delta also inklusive
`updated_on`. Die Zahl ist die erste Klammer um den Lauf.

Der App Token kommt aus `.env` (`SOCRATA_APP_TOKEN`) und wandert in den
Request-Header, nie in die URL — deshalb kann die Manifest-URL aus Schritt 3
gefahrlos abgelegt werden.

In [ ]:
token = dc.app_token(disabled=not USE_TOKEN)

with dc.build_session(token) as session:
    fingerprint, fields = dc.source_schema(session)
    count_before = dc.expected_rows(session, YEAR, updated_from)

print('App Token verwendet        :', token is not None)
print('Felder in der Quelle       :', len(fields))
print('Schema-Fingerabdruck       :', fingerprint)
print('Erwartete Zeilen im Fenster:', f'{count_before:,}' if count_before is not None else None)
print()
print('Auszug aus den Feldtypen der Quelle:')
for name, kind in list(fields.items())[:8]:
    print(f'  {name:24} {kind}')

## Schritt 5 – Lauf ausführen

`download()` und `write_run()` sind genau die beiden Schritte, die auch `main()`
nacheinander aufruft.

**Die Vollständigkeitskette in `download()`** (nur bei `MAX_RECORDS = None`):

1. **Zeilenzahl vorher** — `count(1)` auf dem Fenster.
2. **Feste id-Liste** — `snapshot_ids()` schreibt die Mitgliedschaft des
   Snapshots fest, bevor geladen wird. Sie muss zur Zeilenzahl passen und darf
   keine Dubletten enthalten.
3. **Download mit Keyset-Pagination** — jede Folgeseite fragt ab der zuletzt
   gesehenen `id` (`... AND id > 13324861`, sortiert nach `id`) statt über
   `$offset`. Bei Offset-Paginierung verschieben gleichzeitige Änderungen an der
   Quelle die Seitengrenzen; mit dem Schlüssel als Anker kann das nicht
   passieren.
4. **Abgleich** — die Menge der geladenen ids muss der festgeschriebenen Liste
   **exakt** entsprechen, nicht nur der Anzahl nach. Sonst bricht der Lauf ab und
   veröffentlicht nichts.
5. **Zeilenzahl nachher** — dokumentiert nur, ob sich die Quelle während des
   Laufs bewegt hat (`source_changed_during_run`), und blockiert nicht.

**Das Schreiben in `write_run()`.** Beide Schichten entstehen zuerst in
versteckten `.part`-Verzeichnissen und werden erst nach vollständigem Erfolg
atomar umbenannt — **zuerst Raw, dann Bronze**. `discard_stale_partials()` räumt
zu Beginn auf, was ein harter Abbruch vor mehr als 24 Stunden liegengelassen
hat; veröffentlichte Läufe werden nie angefasst.

In [ ]:
run = None

if RUN_DOWNLOAD:
    started = datetime.now(timezone.utc)
    dc.discard_stale_partials(DATA_ROOT, started)

    result = dc.download(YEAR, updated_from, token, MAX_RECORDS)
    run = dc.write_run(
        DATA_ROOT,
        started,
        result,
        year=YEAR,
        mode=mode,
        old_watermark=watermark,
        updated_from=updated_from,
        token_used=token is not None,
        max_records=MAX_RECORDS,
    )

    print()
    print('Raw    :', run.raw)
    print('Bronze :', run.bronze)
    print('Zeilen :', len(result.records))
    print('Seiten :', len(result.bodies), '(je eine Datei in Raw)')
    print('id-Liste festgeschrieben:', result.snapshot_size)
else:
    print('Download deaktiviert. RUN_DOWNLOAD = True setzen.')

## Schritt 6 – Partitionierung: wie ein Lauf abgelegt wird

Beide Schichten teilen dieselbe `run_id` und folgen demselben Pfadschema im
Hive-Stil (`schlüssel=wert`):

```text
data/
├── raw/
│   └── city=chicago/
│       └── ingest_date=2026-09-06/
│           └── run_20260906T112233444555Z/
│               ├── page-0001.json.gz     ← eine Datei je API-Antwort
│               ├── page-0002.json.gz
│               └── manifest.json
└── bronze/
    └── city=chicago/
        └── ingest_date=2026-09-06/
            └── run_20260906T112233444555Z/
                ├── part-0001.parquet
                └── manifest.json
```

Drei Punkte, die leicht überraschen:

- **`ingest_date` ist das Abrufdatum, nicht das Tatdatum.** Ein Lauf am 6.9.2026
  über den Jahrgang 2024 landet unter `ingest_date=2026-09-06`. Das Tatdatum
  steckt in der Spalte `date` und im `WHERE` aus Schritt 3.
- **`city=chicago` ist fest verdrahtet** (`dc.CITY`) und macht die Ablage offen
  für weitere Städte in derselben Struktur.
- **`run_id` ist ein UTC-Zeitstempel mit Mikrosekunden**, aus dem Startzeitpunkt
  gebildet. Er verbindet Raw und Bronze desselben Laufs; das Bronze-Manifest
  verweist zusätzlich über `derived_from` auf den Raw-Lauf.

Weil `ingest_date` und `city` als `schlüssel=wert` im Pfad stehen, liest PyArrow
sie später als **Partitionsspalten** wieder heraus — dazu Schritt 8.

In [ ]:
if run is not None:
    rel = run.bronze.relative_to(DATA_ROOT)
    print('Bronze-Lauf relativ zum Datenordner:', rel)
    print()
    print('  Schicht    :', rel.parts[0])
    print('  Stadt      :', rel.parts[1])
    print('  Abrufdatum :', rel.parts[2])
    print('  Lauf-ID    :', rel.parts[3])
    print()
    print('Die Dateien dieses Laufs (beide Schichten, dieselbe run_id):')
    for schicht in (run.raw, run.bronze):
        for path in sorted(schicht.rglob('*')):
            if path.is_file():
                print(f'  {path.stat().st_size:>9,} B  {path.relative_to(DATA_ROOT)}')

## Schritt 7 – Das Manifest lesen

Jede Schicht bekommt ihr eigenes, in sich geschlossenes Manifest. Vier Blöcke
sind besonders interessant:

- **`watermark`** — `previous_maximum` (womit der Lauf startete),
  `lookback_days`, `query_from` (der tatsächliche Fensterstart) und
  `maximum_seen` (das neue Wasserzeichen).
- **`completeness`** — `verified` sagt, ob die Snapshot-Kette durchlaufen wurde.
  Bei einem Testlauf steht hier `false`, weil die Gates übersprungen wurden.
- **`source_schema`** — Fingerabdruck und Feldtypen der Quelle zum Zeitpunkt des
  Laufs.
- **`query`** — die erste Seiten-URL ohne Token, `whole_records: true` (es wird
  kein `$select` benutzt) und `dropped_field_prefix`.

**Wie `maximum_seen` entsteht:** aus dem Maximum über das *alte* Wasserzeichen
und alle `updated_on` der geladenen Zeilen. Das alte ist bewusst mit im Topf —
dadurch kann das Wasserzeichen **nie zurückfallen**, auch wenn ein Delta-Lauf
null Zeilen liefert. Verglichen wird als Text; das stimmt nur, weil die API feste
Breite liefert (`2024-05-01T09:00:00.000`).

In [ ]:
if run is not None:
    manifest = json.loads((run.bronze / 'manifest.json').read_text(encoding='utf-8'))

    print('status / mode :', manifest['status'], '/', manifest['mode'])
    print('row_count     :', manifest['row_count'])
    print('max_records   :', manifest['max_records'], '(gesetzt = Testlauf)')
    print()
    for block in ('watermark', 'completeness', 'query'):
        print(f'--- {block} ---')
        print(json.dumps(manifest[block], indent=2, ensure_ascii=False))
        print()
    print('--- source_schema (nur Fingerabdruck) ---')
    print(manifest['source_schema']['fingerprint'])

## Schritt 8 – Bronze lesen: festes Schema, `_extra` und die Partitionsspalten

**Das Schema ist gepinnt.** 22 bekannte Felder plus `_extra` ergeben 23 Spalten,
und zwar in jedem Lauf, den es je gab. Die Werte behalten den JSON-Typ der API:
`arrest` und `domestic` sind `bool`, `location` ist ein `struct`, alle übrigen
19 Felder sind `string` — auch Datum, Zahlen und Koordinaten. Geparst wird erst
oberhalb von Bronze.

**`_extra` fängt alles auf, was nicht ins Schema passt**, als JSON-Text:

- ein Feld, das die Stadt später ergänzt, etwa `{"victim_age":"34"}`
- ein Wert, dessen JSON-Typ sich ändert, etwa `{"arrest":"true"}` — die Spalte
  `arrest` bleibt für diese Zeile leer
- ein `location`-Objekt mit unerwarteten Schlüsseln

**Socrata-eigene Spalten werden verworfen.** Alles im `:`-Namensraum
(`:@computed_region_*`, `:id`, `:updated_at`) sind Plattform-Artefakte, keine
Daten der Stadt — sie landen nicht einmal in `_extra`.

**Und die Falle beim Lesen:** `pq.read_table()` geht über die Dataset-Schicht.
Die wertet die Hive-Verzeichnisse über der Datei aus und hängt `city` und
`ingest_date` als zusätzliche Spalten an — je nach PyArrow-Version auch dann,
wenn nur eine einzelne Datei übergeben wird. Wer genau das Schema der Datei
sehen will, liest sie direkt über `pq.ParquetFile()`.

In [ ]:
import pyarrow.parquet as pq

if run is not None:
    part = run.bronze / dc.PARQUET_FILE

    # Genau das, was in der Datei steht - ohne Dataset-Schicht.
    schema = pq.ParquetFile(part).schema_arrow
    print(f'Spalten in der Datei: {len(schema)}')
    for field in schema:
        print(f'  {field.name:22} {field.type}')
    print()
    print('Entspricht dem gepinnten Schema:', schema.equals(dc.SCHEMA))
    print()

    table = pq.ParquetFile(part).read()
    mit_extra = [value for value in table.column(dc.EXTRA_COLUMN).to_pylist() if value]
    print(f'_extra befüllt in {len(mit_extra)} von {table.num_rows} Zeilen')
    for value in mit_extra[:3]:
        print('  ', value)
    print()

    # Zum Vergleich: über die Dataset-Schicht kommen die Partitionsspalten dazu.
    via_dataset = [field.name for field in pq.read_table(part).schema]
    zusatzspalten = [name for name in via_dataset if name not in dc.SCHEMA.names]
    print('Zusatzspalten via pq.read_table():',
          zusatzspalten or 'keine (neuere PyArrow-Version)')
    print()

# Solange die Quelle nichts Unerwartetes liefert, bleibt _extra leer. Was
# passieren würde, zeigt bronze_row() direkt - dieselbe Funktion, die jede
# Zeile nach Bronze abbildet:
print('Ein neues Feld der Stadt:')
print('  _extra =', dc.bronze_row({'id': '1', 'victim_age': '34'})[dc.EXTRA_COLUMN])

print('Ein Wert, dessen JSON-Typ sich ändert:')
zeile = dc.bronze_row({'id': '2', 'arrest': 'true'})
print('  arrest =', zeile['arrest'], '-> Spalte bleibt leer')
print('  _extra =', zeile[dc.EXTRA_COLUMN])

print('Eine Socrata-eigene Spalte:')
print('  _extra =', dc.bronze_row({'id': '3', ':@computed_region_x': '42'})[dc.EXTRA_COLUMN],
      '-> verworfen, nicht aufbewahrt')

## Schritt 9 – Was der nächste Lauf tun würde

Dieselbe Rechnung wie Schritt 3, nur jetzt mit dem gerade geschriebenen Lauf im
Datenordner. Damit lässt sich prüfen, ob das Wasserzeichen vorgerückt ist.

**Bei einem Testlauf bleibt es bewusst bei `None`:** `previous_watermark()`
überspringt Läufe mit gesetztem `max_records`. Der nächste Lauf wäre also
weiterhin ein Vollabzug — genau so soll es sein, denn ein gekürzter Lauf hat den
Jahrgang nie vollständig gesehen.

Erst ein Lauf mit `MAX_RECORDS = None` setzt das Wasserzeichen, und der Lauf
danach wird zum Delta mit 14 Tagen Überlappung.

In [ ]:
wasserzeichen_jetzt = dc.previous_watermark(DATA_ROOT, YEAR)
fenster_start = dc.delta_start(wasserzeichen_jetzt) if wasserzeichen_jetzt else None

print('Wasserzeichen im Datenordner:', wasserzeichen_jetzt)
print('Nächster Lauf ab            :', fenster_start)
print('Nächster Modus              :', 'delta' if fenster_start else 'full_snapshot')

if MAX_RECORDS is not None:
    print()
    print('Hinweis: Testlauf - er wird als Wasserzeichen ignoriert (max_records gesetzt).')

## Schritt 10 – Aufräumen

**Was die Pipeline selbst aufräumt.** Während eines Laufs entstehen versteckte
Arbeitsverzeichnisse `.<run_id>.part`. Scheitert der Lauf — auch bei
`KeyboardInterrupt`, weil `write_run()` auf `BaseException` fängt —, werden sie
sofort wieder entfernt. Es bleibt also nichts Halbfertiges liegen. Nur ein
harter Abbruch (Prozess-Kill, Stromausfall) kommt nicht mehr zum Aufräumen; was
dabei liegenbleibt, entfernt `discard_stale_partials()` beim nächsten Start,
sobald es älter als `STALE_PARTIAL_HOURS` (24 Stunden) ist. Veröffentlichte
Läufe werden dabei nie angefasst.

**Was liegen bleibt.** Die *veröffentlichten* Läufe — das ist Absicht, sie sind
unveränderliche Beweisstücke. Und der **Testordner** aus Schritt 2 unter
`$TMPDIR/chicago-ingestion-test`: Den räumt niemand automatisch ab. macOS leert
`/var/folders` zwar irgendwann selbst (spätestens beim Neustart), verlassen
sollte man sich darauf nicht. Jeder Notebook-Durchlauf legt dort einen weiteren
Lauf ab.

Die Zelle löscht deshalb den Testordner — und rührt den echten Datenordner
`data/` grundsätzlich nicht an. `CLEANUP = False` setzen, wer die Testdaten
vorher noch anschauen möchte.

In [ ]:
CLEANUP = True   # auf False setzen, um die Testdaten zum Anschauen zu behalten

if DATA_ROOT == PROJECT_ROOT / 'data':
    print('DATA_ROOT ist der echte Datenordner - hier wird nichts gelöscht.')
elif not CLEANUP:
    print('Testdaten bleiben liegen:', DATA_ROOT)
elif DATA_ROOT.exists():
    manifeste = len(list(DATA_ROOT.rglob('manifest.json')))
    shutil.rmtree(DATA_ROOT)
    print(f'Testordner entfernt ({manifeste} Manifest(e)):', DATA_ROOT)
else:
    print('Nichts aufzuräumen.')

# Zur Kontrolle: liegengebliebene Arbeitsverzeichnisse gäbe es hier.
reste = [p for p in DATA_ROOT.glob('*/city=*/ingest_date=*/.*.part')] if DATA_ROOT.exists() else []
print('Offene .part-Verzeichnisse:', reste or 'keine')

## Ergebnis

Nach einem erfolgreichen Lauf liegen zwei Verzeichnisse mit derselben `run_id`
vor:

- `raw/city=chicago/ingest_date=<datum>/<run_id>/` — die API-Antworten Byte für
  Byte, gzip-komprimiert mit `mtime=0`, damit der SHA-256 im Manifest allein von
  der Antwort abhängt. Raw ist das Beweisstück und lässt sich nicht
  rekonstruieren.
- `bronze/city=chicago/ingest_date=<datum>/<run_id>/` — daraus abgeleitetes
  Parquet mit festem Schema, jederzeit neu erzeugbar.

### Die Spezialitäten auf einen Blick

| Thema | Wie es gelöst ist |
| --- | --- |
| Delta ohne Zustand | Wasserzeichen wird aus den Bronze-Manifesten rekonstruiert |
| Rückwirkende Korrekturen | 14 Tage Lookback, Duplikate bewusst in Kauf genommen |
| Wasserzeichen fällt nie zurück | Maximum über altes Wasserzeichen **und** neue Zeilen |
| Testläufe | über `max_records` markiert und als Wasserzeichen ignoriert |
| Vollständigkeit | feste id-Liste vor dem Download, exakter Abgleich danach |
| Seitengrenzen | Keyset-Pagination über `id` statt `$offset` |
| Schema-Drift | SHA-256 über die Feldtypen der Quelle im Manifest |
| Neue API-Felder | landen unverändert in `_extra`, statt verloren zu gehen |
| Abbruch mitten im Lauf | `.part`-Verzeichnisse, atomares Umbenennen, Raw zuerst |
| Liegengebliebene Reste | werden nach 24 Stunden beim nächsten Start entfernt |

### Weiter

Ein echter Lauf über den ganzen Jahrgang: `MAX_RECORDS = None` setzen und das
Notebook erneut ausführen — oder auf der Kommandozeile:

```bash
python -m chicago_pipeline.download_chicago --year 2024
```

Die Bronze-Schicht ist anschliessend der Input für die Staging-Schicht, in der
Zeitstempel geparst, Typen gecastet und mehrere Versionen derselben `id`
aufgelöst werden.